# Семинар 7 [17-25-2]. Схема Бернулли, формулы Лапласа и Пуассона: симуляции

В этом ноутбуке:

1. формулы Бернулли, Лапласа и Пуассона в коде и проверка задач S07-01 – S07-04;
2. виртуальная монетка: как из случайных бросков складывается распределение;
3. данные нашей группы против симуляции и формул;
4. насколько «необычна» наша группа;
5. карта: где работает Лаплас, а где Пуассон;
6. редкие события и формула Пуассона;
7. сколько бросков нужно, чтобы поймать нечестную монету.

Запускайте ячейки по порядку: **Среда выполнения → Выполнить все** или `Shift+Enter` по одной.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(2026)   # фиксируем зерно, чтобы у всех получались одинаковые картинки
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Формулы в коде

| Формула | Что считает | Условия |
|---|---|---|
| Бернулли $P_n(m) = C_n^m p^m q^{n-m}$ | ровно $m$ успехов, точно | любое $n$ (руками — небольшое) |
| Локальная Лапласа $P_n(m) \approx \frac{1}{\sqrt{npq}}\varphi(x)$ | ровно $m$ успехов, приближённо | $n$ велико, $p$ не у края |
| Интегральная Лапласа $P_n(m_1,m_2) \approx \Phi(x_2) - \Phi(x_1)$ | от $m_1$ до $m_2$ | $n$ велико, $p$ не у края |
| Пуассон $P_n(m) \approx \frac{\lambda^m}{m!}e^{-\lambda}$, $\lambda=np$ | ровно $m$ редких событий | $n$ велико, $p$ мало, $np<10$ |

In [ ]:
def bernoulli_pmf(n, m, p):
    # Формула Бернулли: ровно m успехов в n испытаниях
    return math.comb(n, m) * p**m * (1 - p)**(n - m)

def phi(x):
    # Плотность стандартного нормального закона φ(x)
    return math.exp(-x**2 / 2) / math.sqrt(2 * math.pi)

def Phi(x):
    # Функция Лапласа Ф(x) = интеграл от 0 до x (нечётная, Ф(∞) = 0,5)
    return 0.5 * math.erf(x / math.sqrt(2))

def laplace_local(n, m, p):
    s = math.sqrt(n * p * (1 - p))
    return phi((m - n * p) / s) / s

def laplace_integral(n, m1, m2, p):
    s = math.sqrt(n * p * (1 - p))
    return Phi((m2 - n * p) / s) - Phi((m1 - n * p) / s)

def poisson_pmf(n, m, p):
    lam = n * p
    return lam**m * math.exp(-lam) / math.factorial(m)

def bernoulli_range(n, m1, m2, p):
    # Точная вероятность «от m1 до m2 успехов» (scipy аккуратно считает и при больших n)
    return stats.binom.cdf(m2, n, p) - stats.binom.cdf(m1 - 1, n, p)

### Проверяем задачи семинара

Сравним точный ответ (Бернулли, посчитанный компьютером) с приближённым ответом по формуле, которой мы решали задачу у доски.

In [ ]:
rows = [
    ("S07-01а: ровно 5 из 6, p=0,8",          bernoulli_pmf(6, 5, 0.8),               bernoulli_pmf(6, 5, 0.8),       "Бернулли"),
    ("S07-01б: не менее 5 из 6",              bernoulli_range(6, 5, 6, 0.8),          bernoulli_range(6, 5, 6, 0.8),  "Бернулли"),
    ("S07-01в: хотя бы одно",                 1 - 0.2**6,                             1 - 0.2**6,                     "Бернулли"),
    ("S07-02: ровно 200 из 400, p=0,5",       stats.binom.pmf(200, 400, 0.5),         laplace_local(400, 200, 0.5),   "Лаплас лок."),
    ("S07-03а: от 792 до 828 из 900, p=0,9",  bernoulli_range(900, 792, 828, 0.9),    laplace_integral(900, 792, 828, 0.9), "Лаплас инт."),
    ("S07-03б: не менее 800 из 900",          bernoulli_range(900, 800, 900, 0.9),    laplace_integral(900, 800, 900, 0.9), "Лаплас инт."),
    ("S07-04а: ровно 3 из 500, p=0,004",      bernoulli_pmf(500, 3, 0.004),           poisson_pmf(500, 3, 0.004),     "Пуассон"),
    ("S07-04б: хотя бы одно",                 1 - 0.996**500,                         1 - math.exp(-2),               "Пуассон"),
]
df = pd.DataFrame(rows, columns=["Задача", "Точно (Бернулли)", "По формуле", "Формула"])
df["Ошибка, %"] = 100 * (df["По формуле"] - df["Точно (Бернулли)"]).abs() / df["Точно (Бернулли)"]
df.round(4)

**Обратите внимание на S07-03а:** интегральная формула Лапласа дала 0,9545, а точный ответ — 0,9604. Ошибка почти в процент. Причина: формула заменяет сумму по *целым* $m$ площадью под гладкой кривой и «теряет» по половинке крайних столбиков. Если взять границы $791{,}5$ и $828{,}5$ (так называемая *поправка на непрерывность*), совпадение станет почти идеальным — проверьте сами в ячейке ниже.

In [ ]:
print("Без поправки:", round(laplace_integral(900, 792, 828, 0.9), 4))
print("С поправкой: ", round(laplace_integral(900, 791.5, 828.5, 0.9), 4))
print("Точно:       ", round(bernoulli_range(900, 792, 828, 0.9), 4))

## 2. Виртуальная монетка

Одна **серия** = 10 бросков монеты, считаем число орлов $m$. Компьютер повторит то, что мы делали руками, но гораздо больше раз.

In [ ]:
N_TOSSES = 10
P_HEADS = 0.5

def simulate_series(n_series, n=N_TOSSES, p=P_HEADS):
    # Каждая строка — серия из n бросков (1 = орёл); возвращаем число орлов в каждой серии
    tosses = rng.random((n_series, n)) < p
    return tosses.sum(axis=1)

print("Пять серий, как у одного студента:", simulate_series(5))

In [ ]:
m_values = np.arange(N_TOSSES + 1)
p_bern = np.array([bernoulli_pmf(N_TOSSES, m, P_HEADS) for m in m_values])

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, n_series in zip(axes, [10, 100, 100_000]):
    counts = simulate_series(n_series)
    freq = np.bincount(counts, minlength=N_TOSSES + 1) / n_series
    ax.bar(m_values, freq, alpha=0.6, label="симуляция")
    ax.plot(m_values, p_bern, "o-", color="black", label="Бернулли")
    ax.set_title(f"{n_series:,} серий".replace(",", " "))
    ax.set_xlabel("число орлов m")
axes[0].set_ylabel("частота")
axes[0].legend()
plt.tight_layout()
plt.show()

**Вопрос:** почему при 10 сериях гистограмма «скачет», а при 100 000 почти совпадает с формулой? Как это связано с тем, что ваши личные 5 серий были не похожи на теорию?

## 3. Данные нашей группы

Впишите в словарь `group_tally` число палочек с доски для каждого $m$ (сколько серий дали $m$ орлов). Если оставить все нули, ноутбук подставит демонстрационные данные из симуляции.

In [ ]:
# ← ВПИШИТЕ СЮДА ДАННЫЕ С ДОСКИ
group_tally = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0}

group = np.array([group_tally[m] for m in m_values])
if group.sum() == 0:
    print("Данных нет — используем демонстрационные: 100 случайных серий.")
    group = np.bincount(simulate_series(100), minlength=N_TOSSES + 1)
N_group = group.sum()
print("Всего серий в группе:", N_group, "| всего бросков:", N_group * N_TOSSES)

In [ ]:
sim_big = np.bincount(simulate_series(100_000), minlength=N_TOSSES + 1) / 100_000
p_lapl = np.array([laplace_local(N_TOSSES, m, P_HEADS) for m in m_values])
p_pois = np.array([poisson_pmf(N_TOSSES, m, P_HEADS) for m in m_values])

table = pd.DataFrame({
    "m": m_values,
    "группа (палочки)": group,
    "группа (частота)": group / N_group,
    "симуляция 100 000": sim_big,
    "Бернулли": p_bern,
    "Лаплас": p_lapl,
    "Пуассон λ=5": p_pois,
}).set_index("m")
display(table.round(4))

plt.bar(m_values, group / N_group, alpha=0.5, label=f"наша группа ({N_group} серий)")
plt.plot(m_values, sim_big, "s", ms=9, mfc="none", label="симуляция 100 000 серий")
plt.plot(m_values, p_bern, "o-", color="black", label="Бернулли")
plt.plot(m_values, p_lapl, "--", label="Лаплас (локальная)")
plt.plot(m_values, p_pois, ":", lw=2.5, label="Пуассон, λ = 5")
plt.xlabel("число орлов в серии m")
plt.ylabel("вероятность / частота")
plt.legend()
plt.show()

In [ ]:
# Одна цифра на каждую модель: насколько далеко частоты группы от неё в сумме
for name, col in [("Бернулли", p_bern), ("Лаплас", p_lapl), ("Пуассон", p_pois)]:
    dist = np.abs(group / N_group - col).sum()
    print(f"Суммарное расхождение группы с моделью «{name}»: {dist:.3f}")
print(f"Пуассон «теряет» вероятность на m > 10: {1 - p_pois.sum():.4f}")

**Вопросы:**

1. Почему Лаплас почти совпадает с Бернулли даже при $n = 10$?
2. Почему Пуассон смещён и несимметричен? Какое условие его применения нарушено?
3. Что означает, что Пуассон даёт ненулевую вероятность выпасть 11 орлам из 10 бросков?

## 4. Насколько «необычна» наша группа?

Посмотрим на общую долю орлов во всех бросках группы. Симулируем 10 000 «виртуальных групп» такого же размера и проверим, как часто случайно получается такое же или ещё большее отклонение от 0,5.

In [ ]:
our_share = (group * m_values).sum() / (N_group * N_TOSSES)
n_total = N_group * N_TOSSES

virtual = rng.binomial(n_total, P_HEADS, size=10_000) / n_total   # 10 000 групп, по n_total бросков
as_extreme = np.mean(np.abs(virtual - 0.5) >= abs(our_share - 0.5))

plt.hist(virtual, bins=40, alpha=0.6, density=True, label="10 000 виртуальных групп")
plt.axvline(our_share, color="red", lw=2, label=f"наша группа: {our_share:.3f}")
s = math.sqrt(P_HEADS * (1 - P_HEADS) / n_total)
xs = np.linspace(virtual.min(), virtual.max(), 200)
plt.plot(xs, stats.norm.pdf(xs, 0.5, s), color="black", label="кривая Лапласа")
plt.xlabel("доля орлов")
plt.legend()
plt.show()
print(f"Доля орлов в группе: {our_share:.3f}")
print(f"Виртуальных групп с таким же или большим отклонением от 0,5: {as_extreme:.1%}")
print(f"Интегральная формула Лапласа: 95 % групп попадают в 0,5 ± {1.96 * s:.3f}")

**Как читать:** если таких же «странных» виртуальных групп много (скажем, больше 5 %), наше отклонение — обычная случайность, и подозревать монеты не в чем.

## 5. Карта: где работает Лаплас, а где Пуассон

Для каждой пары $(n, p)$ считаем максимальную по $m$ ошибку приближения: $\max_m |P_\text{прибл}(m) - P_\text{Бернулли}(m)|$. Тёмное — приближение хорошее, светлое — плохое.

In [ ]:
ns = np.array([10, 20, 50, 100, 200, 500, 1000])
ps = np.array([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5])

def max_error(n, p, kind):
    m = np.arange(n + 1)
    exact = stats.binom.pmf(m, n, p)
    if kind == "laplace":
        s = math.sqrt(n * p * (1 - p))
        approx = stats.norm.pdf((m - n * p) / s) / s
    else:
        approx = stats.poisson.pmf(m, n * p)
    return np.abs(approx - exact).max()

err_l = np.array([[max_error(n, p, "laplace") for p in ps] for n in ns])
err_p = np.array([[max_error(n, p, "poisson") for p in ps] for n in ns])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, err, title in [(axes[0], err_l, "Ошибка локальной формулы Лапласа"),
                       (axes[1], err_p, "Ошибка формулы Пуассона")]:
    im = ax.imshow(np.log10(err), cmap="viridis", aspect="auto", vmin=-5, vmax=0)
    ax.set_xticks(range(len(ps)), [str(p) for p in ps], rotation=45)
    ax.set_yticks(range(len(ns)), [str(n) for n in ns])
    ax.set_xlabel("p"); ax.set_ylabel("n"); ax.set_title(title)
    for i in range(len(ns)):
        for j in range(len(ps)):
            ax.text(j, i, f"{err[i, j]:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if err[i, j] < 0.01 else "black")
    ax.grid(False)
fig.colorbar(im, ax=axes, label="log10(ошибка)")
plt.show()

**Что видно на карте:**

- Лаплас становится точным, когда растёт $n$ и $p$ ближе к 0,5 (правый нижний угол левой карты); при маленьком $p$ он плох даже при $n = 1000$.
- Пуассон точен при маленьком $p$ (левая часть правой карты) и почти не зависит от $n$; при $p = 0{,}5$ он не годится никогда.
- Приближения дополняют друг друга: там, где плох один, хорош другой.

## 6. Редкие события: формула Пуассона в деле

Партия из $n = 1000$ семян, вероятность того, что семя заражено, $p = 0{,}003$. Сколько заражённых семян в партии? Сравним симуляцию 100 000 партий с Бернулли, Пуассоном и Лапласом.

In [ ]:
n, p = 1000, 0.003
batches = rng.binomial(n, p, size=100_000)
m = np.arange(0, 12)
freq = np.bincount(batches, minlength=12)[:12] / len(batches)
s = math.sqrt(n * p * (1 - p))

plt.bar(m, freq, alpha=0.5, label="симуляция 100 000 партий")
plt.plot(m, stats.binom.pmf(m, n, p), "o-", color="black", label="Бернулли")
plt.plot(m, stats.poisson.pmf(m, n * p), "s--", mfc="none", label=f"Пуассон, λ = {n * p:g}")
plt.plot(m, stats.norm.pdf((m - n * p) / s) / s, ":", lw=2.5, label="Лаплас")
plt.xlabel("заражённых семян в партии"); plt.ylabel("вероятность")
plt.legend(); plt.show()

**Вопрос:** почему здесь Лаплас «залезает» в отрицательную область и недооценивает $m = 0$, а Пуассон почти совпадает с Бернулли?

## 7. Сколько бросков нужно, чтобы поймать нечестную монету?

Исследование 2023 года (Bartoš et al., 350 757 бросков) показало, что монета падает той же стороной, какой лежала до броска, примерно в 50,8 % случаев. Пусть у нас есть такая монета. При каком числе бросков мы заметим, что она нечестная?

Правило: считаем монету «подозрительной», если доля орлов выходит за пределы $0{,}5 \pm 1{,}96\sqrt{0{,}25/n}$ (95 % по интегральной формуле Лапласа).

In [ ]:
p_true = 0.508
sizes = [100, 500, 2_000, 10_000, 50_000, 350_000]
caught = []
for n in sizes:
    shares = rng.binomial(n, p_true, size=5_000) / n
    caught.append(np.mean(np.abs(shares - 0.5) > 1.96 * math.sqrt(0.25 / n)))

plt.semilogx(sizes, caught, "o-")
plt.axvline(500, color="red", ls="--", label="наш семинар (~500 бросков)")
plt.xlabel("число бросков n"); plt.ylabel("доля экспериментов, поймавших нечестность")
plt.legend(); plt.show()
for n, c in zip(sizes, caught):
    print(f"n = {n:>7}: монету поймали в {c:.0%} экспериментов")

**Вывод:** на 500 бросках отличить 50,8 % от 50 % практически невозможно: формулы и симуляция говорят одно и то же. Именно поэтому исследователям понадобились сотни тысяч бросков.

---

## Итоги

1. Формула Бернулли — точная, Лаплас и Пуассон — приближения при большом $n$.
2. Лаплас хорош, когда $p$ не прижато к краю и $npq$ велико; Пуассон — когда $p$ мало (редкие события).
3. Отдельная серия случайна, но тысячи серий складываются в распределение, которое и описывают формулы.